In [1]:
# ============ START: inventory.py ============
import os
import wfdb
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Root folder containing all 8 dataset subfolders
DATA_ROOT = Path(r"C:\ECG_Project\training")

# The 8 hospital source folders you downloaded
SOURCE_FOLDERS = [
    "chapman_shaoxing",
    "cpsc_2018",
    "cpsc_2018_extra",
    "georgia",
    "ningbo",
    "ptb-xl"
]


def read_header_metadata(hea_path: Path, source: str) -> dict | None:
    """
    Reads a single .hea file and extracts the metadata we need,
    WITHOUT loading the actual signal (fast, low memory).
    Returns None if the file is broken/unreadable (so we can skip it later).
    """
    try:
        record_id = hea_path.stem
        header = wfdb.rdheader(str(hea_path.with_suffix("")))

        age, sex, dx_codes = None, None, None
        for comment in header.comments:
            if comment.startswith("Age:"):
                age = comment.split("Age:")[1].strip()
            elif comment.startswith("Sex:"):
                sex = comment.split("Sex:")[1].strip()
            elif comment.startswith("Dx:"):
                dx_codes = comment.split("Dx:")[1].strip()

        return {
            "record_id": record_id,
            "source_hospital": source,
            "sampling_rate": header.fs,
            "num_leads": header.n_sig,
            "num_samples": header.sig_len,
            "lead_names": ",".join(header.sig_name),
            "age": age,
            "sex": sex,
            "dx_codes": dx_codes,
            "file_path": str(hea_path.with_suffix("")),
        }
    except Exception as e:
        print(f"  [SKIPPED] {hea_path.name}: {e}")
        return None


def build_inventory():
    all_records = []
    for source in SOURCE_FOLDERS:
        folder_path = DATA_ROOT / source
        if not folder_path.exists():
            print(f"Folder not found, skipping: {folder_path}")
            continue

        hea_files = list(folder_path.rglob("*.hea"))
        print(f"{source}: found {len(hea_files)} header files")

        for hea_path in tqdm(hea_files, desc=source):
            record = read_header_metadata(hea_path, source)
            if record is not None:
                all_records.append(record)

    print(f"\nTotal records successfully read: {len(all_records)}")

    metadata_df = pd.DataFrame(all_records)
    print(metadata_df.shape)
    print(metadata_df.head())

    out_path = DATA_ROOT / "ecg_metadata_inventory.csv"
    metadata_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

    return metadata_df


if __name__ == "__main__":
    build_inventory()
# ============ END: inventory.py ============

chapman_shaoxing: found 10229 header files


chapman_shaoxing: 100%|██████████████████████████████████████████████████████████| 10229/10229 [04:20<00:00, 39.23it/s]


cpsc_2018: found 6877 header files


cpsc_2018: 100%|███████████████████████████████████████████████████████████████████| 6877/6877 [02:47<00:00, 41.07it/s]


cpsc_2018_extra: found 3452 header files


...

ningbo: found 34902 header files


ningbo: 100%|████████████████████████████████████████████████████████████████████| 34902/34902 [15:04<00:00, 38.61it/s]


ptb-xl: found 21836 header files


ptb-xl: 100%|████████████████████████████████████████████████████████████████████| 21836/21836 [08:34<00:00, 42.44it/s]



Total records successfully read: 87638
(87638, 10)
  record_id   source_hospital  sampling_rate  num_leads  num_samples  \
0   JS00001  chapman_shaoxing            500         12         5000   
1   JS00002  chapman_shaoxing            500         12         5000   
2   JS00004  chapman_shaoxing            500         12         5000   
3   JS00005  chapman_shaoxing            500         12         5000   
4   JS00006  chapman_shaoxing            500         12         5000   

                               lead_names age     sex  \
0  I,II,III,aVR,aVL,aVF,V1,V2,V3,V4,V5,V6  85    Male   
1  I,II,III,aVR,aVL,aVF,V1,V2,V3,V4,V5,V6  59  Female   
2  I,II,III,aVR,aVL,aVF,V1,V2,V3,V4,V5,V6  66    Male   
3  I,II,III,aVR,aVL,aVF,V1,V2,V3,V4,V5,V6  73  Female   
4  I,II,III,aVR,aVL,aVF,V1,V2,V3,V4,V5,V6  46  Female   

                        dx_codes  \
0   164889003,59118001,164934002   
1            426177001,164934002   
2                      426177001   
3  164890007,429622005,42875